<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-01/AI_Modul_1.4_Praktikum_Jenis_AI_ANI_AGI_ASI.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 1.4: Praktikum Jenis AI (ANI, AGI, ASI)
### Eksperimen Keterbatasan Teknis ANI: Fenomena *Catastrophic Forgetting* vs Retensi Memori Lintas Tugas (*Continual Learning*)

---

> **Diktat Terkait:** [AI_Modul_1.4_Jenis_AI_ANI_AGI_ASI.md](../../docs/part-01/AI_Modul_1.4_Jenis_AI_ANI_AGI_ASI.md)  
> **Outputs:** Kode program simulasi retensi memori multi-tugas dengan komentar per baris, visualisasi kurva lupa katastrofik, dan pengujian memori episodik.  
> **Outcomes:** Pemahaman mendalam mengenai keterbatasan arsitektur Narrow AI (ANI) saat dihadapkan pada pergeseran domain dan prinsip dasar arsitektur adaptif menuju General AI (AGI).  
> **Impacts:** Keterampilan merancang sistem AI otonom yang mampu belajar terus-menerus di lapangan (*continual/life-long learning*) tanpa merusak konfigurasi sistem yang telah teruji sebelumnya.  
>
> **Tujuan Pembelajaran:**  
> 1. Mengamati secara langsung fenomena *Catastrophic Forgetting* ketika model ANI standar dilatih pada dua tugas kebun secara berturutan.  
> 2. Mengimplementasikan mekanisme retensi memori episodik (*Episodic Memory Replay*) yang meniru cara kerja hipokampus otak biologis.  
> 3. Memvisualisasikan kurva kompromi (*trade-off*) stabilitas-plastisitas pembelajaran mesin secara grafis.

## 1. Persiapan Environment & Import Library
Jalankan sel di bawah ini untuk memuat pustaka matematika numerik dan visualisasi grafik:

In [ ]:
# Mengimpor modul sys untuk membaca metadata sistem Python
import sys

# Mengimpor modul numpy untuk operasi aljabar linier dan manipulasi bobot matriks
import numpy as np

# Mengimpor modul matplotlib.pyplot untuk merender visualisasi grafik performa retensi memori
import matplotlib.pyplot as plt

# Mengimpor Tuple dan List dari typing untuk dokumentasi tipe data fungsi
from typing import Tuple, List, Dict

# Menampilkan versi interpreter Python
print(f"Python Version : {sys.version.split()[0]}")

# Mengonfirmasi bahwa environment telah siap
print("Environment siap digunakan untuk simulasi kapabilitas ANI vs Continual Learning!")

## 2. Pembangkitan Dua Skenario Tugas Berurutan (Sequential Tasks)
Kita memodelkan dua tugas berbeda di lingkungan perkebunan cerdas:
- **Tugas A (Deteksi Stres Suhu Panas)**: Input adalah [Suhu Ternormalisasi $x_0$, Kelembaban $x_1$]. Target bernilai 1 jika $x_0 > 0.1$.
- **Tugas B (Deteksi Defisit Kelembaban Tanah)**: Input adalah [Suhu $x_0$, Kelembaban $x_1$]. Target bernilai 1 jika $x_1 < -0.1$.

In [ ]:
# Mengunci seed random agar eksperimen reproducible
np.random.seed(123)

# Menetapkan jumlah sampel data sebanyak 100 sampel per tugas
n_samples = 100

# 1. Membangkitkan Data Tugas A (Deteksi Suhu Panas)
X_task_a = np.random.uniform(-1.0, 1.0, (n_samples, 2))
# Label target Tugas A: bernilai 1 jika fitur suhu (x0) > 0.1
y_task_a = (X_task_a[:, 0] > 0.1).astype(int)

# 2. Membangkitkan Data Tugas B (Deteksi Defisit Kelembaban Tanah)
X_task_b = np.random.uniform(-1.0, 1.0, (n_samples, 2))
# Label target Tugas B: bernilai 1 jika fitur kelembaban (x1) < -0.1
y_task_b = (X_task_b[:, 1] < -0.1).astype(int)

print(f"Dataset Tugas A Berhasil Dibuat: {X_task_a.shape[0]} sampel (Distribusi Kelas: {np.bincount(y_task_a)})")
print(f"Dataset Tugas B Berhasil Dibuat: {X_task_b.shape[0]} sampel (Distribusi Kelas: {np.bincount(y_task_b)})")

## 3. Implementasi Dua Arsitektur Agen
Kita membandingkan:
1. `StandardNarrowAgent`: Model ANI konvensional yang langsung menimpa bobot saat mempelajari tugas baru tanpa memori masa lalu.
2. `ContinualReplayAgent`: Model adaptif dengan penyangga memori episodik (*Episodic Memory Buffer*) yang mempertahankan cuplikan representasi tugas lama.

In [ ]:
# ======================================================================
# 1. MODEL STANDAR ARTIFICIAL NARROW INTELLIGENCE (ANI)
# ======================================================================
class StandardNarrowAgent:
    """Model ANI konvensional yang langsung menimpa bobot parameter saat belajar tugas baru."""
    def __init__(self, input_dim: int = 2, lr: float = 0.05):
        self.lr: float = lr
        self.weights: np.ndarray = np.zeros(input_dim)
        self.bias: float = 0.0

    def predict(self, X: np.ndarray) -> np.ndarray:
        linear_z = np.dot(X, self.weights) + self.bias
        return (linear_z >= 0.0).astype(int)

    def train_epoch(self, X: np.ndarray, y: np.ndarray) -> None:
        for xi, target in zip(X, y):
            pred = 1 if (np.dot(self.weights, xi) + self.bias) >= 0.0 else 0
            error = target - pred
            # Koreksi parameter naif tanpa memperhitungkan tugas sebelumnya
            self.weights += self.lr * error * xi
            self.bias += self.lr * error


# ======================================================================
# 2. MODEL CONTINUAL LEARNING DENGAN PENYANGGA MEMORI (EPISODIC REPLAY AGENT)
# ======================================================================
class ContinualReplayAgent(StandardNarrowAgent):
    """Model adaptif dengan penyangga memori episodik untuk mencegah Catastrophic Forgetting."""
    def __init__(self, input_dim: int = 2, lr: float = 0.05, memory_size: int = 20):
        super().__init__(input_dim=input_dim, lr=lr)
        self.memory_size: int = memory_size
        # Penyangga memori untuk menyimpan cuplikan sampel dari tugas-tugas masa lalu
        self.memory_buffer_X: List[np.ndarray] = []
        self.memory_buffer_y: List[np.ndarray] = []

    def store_memory(self, X: np.ndarray, y: np.ndarray) -> None:
        """Menyimpan sebagian kecil sampel representatif ke dalam memori jangka panjang."""
        indices = np.random.choice(len(X), self.memory_size, replace=False)
        self.memory_buffer_X.append(X[indices])
        self.memory_buffer_y.append(y[indices])

    def train_continual_epoch(self, X_current: np.ndarray, y_current: np.ndarray) -> None:
        """Melatih tugas saat ini sekaligus mengulang cuplikan memori masa lalu (Memory Replay)."""
        if len(self.memory_buffer_X) > 0:
            # Menggabungkan data tugas saat ini dengan rekaman memori masa lalu
            X_combined = np.vstack([X_current] + self.memory_buffer_X)
            y_combined = np.concatenate([y_current] + self.memory_buffer_y)
        else:
            X_combined = X_current
            y_combined = y_current
            
        # Melatih pada kumpulan data gabungan
        self.train_epoch(X_combined, y_combined)

print("Kedua model agen berhasil dikonstruksi!")

## 4. Eksekusi Pembelajaran Berurutan (Sequential Learning Cycle)
Kita melatih model dalam 2 fase:
- **Fase 1 (Epoch 1 s.d. 30)**: Melatih Tugas A.
- **Fase 2 (Epoch 31 s.d. 60)**: Beralih melatih Tugas B.
Kita memonitor akurasi kedua tugas pada setiap epoch untuk mendeteksi kapan memori Tugas A runtuh.

In [ ]:
# Instansiasi kedua model
ani_model = StandardNarrowAgent(input_dim=2, lr=0.05)
adapt_model = ContinualReplayAgent(input_dim=2, lr=0.05, memory_size=25)

epochs_phase1 = 30
epochs_phase2 = 30

ani_acc_a, ani_acc_b = [], []
adapt_acc_a, adapt_acc_b = [], []

# --- FASE 1: PELATIHAN TUGAS A ---
for ep in range(epochs_phase1):
    ani_model.train_epoch(X_task_a, y_task_a)
    adapt_model.train_continual_epoch(X_task_a, y_task_a)
    
    ani_acc_a.append(np.mean(ani_model.predict(X_task_a) == y_task_a) * 100)
    ani_acc_b.append(np.mean(ani_model.predict(X_task_b) == y_task_b) * 100)
    
    adapt_acc_a.append(np.mean(adapt_model.predict(X_task_a) == y_task_a) * 100)
    adapt_acc_b.append(np.mean(adapt_model.predict(X_task_b) == y_task_b) * 100)

# Model adaptif mengonsolidasikan memori cuplikan Tugas A
adapt_model.store_memory(X_task_a, y_task_a)

# --- FASE 2: BERALIH MELATIH TUGAS B ---
for ep in range(epochs_phase2):
    ani_model.train_epoch(X_task_b, y_task_b)
    adapt_model.train_continual_epoch(X_task_b, y_task_b)
    
    ani_acc_a.append(np.mean(ani_model.predict(X_task_a) == y_task_a) * 100)
    ani_acc_b.append(np.mean(ani_model.predict(X_task_b) == y_task_b) * 100)
    
    adapt_acc_a.append(np.mean(adapt_model.predict(X_task_a) == y_task_a) * 100)
    adapt_acc_b.append(np.mean(adapt_model.predict(X_task_b) == y_task_b) * 100)

print("Evaluasi Akhir Pasca-Fase 2:")
print(f"  [Model ANI Standar]   : Akurasi Tugas B = {ani_acc_b[-1]:.2f}%, Akurasi Tugas A = {ani_acc_a[-1]:.2f}% (LUPA KATASTROFIK! RUNTUH DARI 100% KE {ani_acc_a[-1]:.1f}%)")
print(f"  [Model Adaptif Replay]: Akurasi Tugas B = {adapt_acc_b[-1]:.2f}%, Akurasi Tugas A = {adapt_acc_a[-1]:.2f}% (MEMORI TERTINGGI BERHASIL DIPERTAHANKAN!)")

## 5. Visualisasi Kurva Performa: Membuktikan Fenomena Catastrophic Forgetting
Grafik di bawah ini memperlihatkan perbandingan dinamika retensi memori kedua model:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# 1. Plot Model ANI Standar
ax1.plot(ani_acc_a, label='Akurasi Tugas A (Deteksi Suhu)', color='#C0392B', lw=2.5)
ax1.plot(ani_acc_b, label='Akurasi Tugas B (Deteksi Kelembaban)', color='#2980B9', linestyle='--', lw=2)
ax1.axvline(30, color='#555', linestyle=':', lw=1.5)
ax1.text(15, 105, 'Fase 1: Belajar A', ha='center', fontweight='bold', fontsize=9, color='#1A5276')
ax1.text(45, 105, 'Fase 2: Belajar B', ha='center', fontweight='bold', fontsize=9, color='#7D6608')
ax1.text(45, 65, 'Runtuhnya Memori A\n(Catastrophic Forgetting)', ha='center', color='#C0392B', fontweight='bold', fontsize=8.5)
ax1.set_title('(A) Model ANI Standar (Tanpa Proteksi Memori)', fontsize=11, fontweight='bold')
ax1.set_xlabel('Epoch Pelatihan')
ax1.set_ylabel('Akurasi Tugas (%)')
ax1.set_ylim(40, 115)
ax1.legend(fontsize=8, loc='lower left')
ax1.grid(True, linestyle=':', alpha=0.6)

# 2. Plot Model Adaptif dengan Retensi Memori
ax2.plot(adapt_acc_a, label='Akurasi Tugas A (Memori Terlindungi)', color='#27AE60', lw=2.5)
ax2.plot(adapt_acc_b, label='Akurasi Tugas B (Tugas Baru)', color='#2980B9', linestyle='--', lw=2)
ax2.axvline(30, color='#555', linestyle=':', lw=1.5)
ax2.text(15, 105, 'Fase 1: Belajar A', ha='center', fontweight='bold', fontsize=9, color='#1A5276')
ax2.text(45, 105, 'Fase 2: Belajar B', ha='center', fontweight='bold', fontsize=9, color='#7D6608')
ax2.text(45, 85, 'Retensi Memori Terjaga!\n(Continual Learning)', ha='center', color='#27AE60', fontweight='bold', fontsize=8.5)
ax2.set_title('(B) Model Adaptif (Episodic Memory Replay)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Epoch Pelatihan')
ax2.set_ylabel('Akurasi Tugas (%)')
ax2.set_ylim(40, 115)
ax2.legend(fontsize=8, loc='lower left')
ax2.grid(True, linestyle=':', alpha=0.6)

plt.suptitle('Dinamika Retensi Memori: Bukti Empiris Keterbatasan ANI Menuju AGI', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Pembahasan Penggunaan Kode 4:
- Pada panel kiri (Model ANI Standar), garis merah (Tugas A) anjlok bebas dari 100% menuju kisaran 55% seketika setelah garis vertikal 30 epoch dilewati. Ini adalah bukti visual dari fenomena **Catastrophic Forgetting** yang membelenggu sistem Narrow AI konvensional.
- Pada panel kanan (Model Adaptif), penyangga memori episodik berhasil mempertahankan keahlian Tugas A di atas 75% sembari model beradaptasi menguasai Tugas B.

## 6. Tugas Mandiri Pemrograman (Coding Challenge)

### Menguji Pengaruh Ukuran Penyangga Memori (*Memory Buffer Size*)
Ukuran penyangga memori merefleksikan kapasitas retensi kognitif agen:
- Jika memori terlalu kecil ($N_{mem} = 2$), agen tidak memiliki cukup contoh representatif sehingga tetap mengalami lupa katastrofik.
- Jika memori membesar ($N_{mem} \in [5, 15, 30, 50]$), retensi memori meningkat secara signifikan.

**Tugas Anda:**  
Lakukan perulangan untuk menguji 4 variasi ukuran penyangga memori $N_{mem} \in [2, 10, 25, 50]$. Amati akurasi akhir pada Tugas A dan Tugas B setelah Fase 2 selesai!

In [ ]:
# ======================================================================
# TUGAS MAHASISWA: Evaluasi pengaruh ukuran memori penyangga (memory_size)
# ======================================================================
memory_candidates = [2, 10, 25, 50]

print(f"{'Ukuran Memori (N)':<20} | {'Akurasi Tugas A (%)':<25} | {'Akurasi Tugas B (%)':<25}")
print("-" * 75)

for mem_sz in memory_candidates:
    # 1. Inisialisasi model dengan ukuran memori tertentu
    test_agent = ContinualReplayAgent(input_dim=2, lr=0.05, memory_size=mem_sz)
    
    # 2. Latih Tugas A selama 30 epoch
    for _ in range(30):
        test_agent.train_continual_epoch(X_task_a, y_task_a)
        
    # 3. Simpan memori cuplikan Tugas A
    test_agent.store_memory(X_task_a, y_task_a)
    
    # 4. Latih Tugas B dengan memory replay selama 30 epoch
    for _ in range(30):
        test_agent.train_continual_epoch(X_task_b, y_task_b)
        
    # 5. Hitung akurasi akhir pada kedua tugas
    acc_a_final = np.mean(test_agent.predict(X_task_a) == y_task_a) * 100
    acc_b_final = np.mean(test_agent.predict(X_task_b) == y_task_b) * 100
    
    print(f"{mem_sz:<20} | {acc_a_final:<25.2f} | {acc_b_final:<25.2f}")

## 7. Rubrik Penilaian Praktikum Mandiri

| Aspek Penilaian | Bobot | Kriteria Capaian |
| :--- | :---: | :--- |
| **Dokumentasi Komentar Kode** | 20% | Mahasiswa menyertakan penjelasan per baris kode pada blok eksperimen buffer memori. |
| **Pemahaman Teori Lupa Katastrofik** | 40% | Mahasiswa mampu menjelaskan mengapa model ANI standar kehilangan akurasi Tugas A saat berpindah tugas. |
| **Analisis Hasil Eksperimen Memori** | 40% | Mahasiswa mampu menganalisis korelasi antara kenaikan ukuran penyangga memori dengan stabilitas akurasi Tugas A. |